In [ ]:
import os, io, json, time, random
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from datasets import load_from_disk
from PIL import Image
from sklearn.metrics import roc_auc_score, balanced_accuracy_score

# ---- Hardware Detection (TPU / GPU / CPU) ----
IS_TPU = False
if "PJRT_DEVICE" in os.environ or "TPU_NAME" in os.environ or os.path.exists("/dev/accel0"):
    try:
        import torch_xla
        import torch_xla.core.xla_model as xm
        import torch_xla.distributed.parallel_loader as pl
        device = xm.xla_device()
        IS_TPU = True
        print("TPU Detected:", xm.xla_real_devices([str(device)]))
    except ImportError:
        print("PyTorch XLA not installed. Defaulting to CUDA/CPU.")

if not IS_TPU:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device, "|", torch.cuda.get_device_name(0) if device.type == "cuda" else "No GPU")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

IMG_EXT = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
IMG_COL = "Image"

def find_dirs(top, names, max_depth=12):
    found, base = [], top.rstrip(os.sep).count(os.sep)
    for root, dirs, _ in os.walk(top):
        if root.count(os.sep) - base >= max_depth:
            dirs[:] = []; continue
        for d in list(dirs):
            if d.lower() in names: found.append(os.path.join(root, d))
            if d.lower() in {"ai", "nature"}: dirs.remove(d)
    return found

# saved dataset (subset + splits)
saved = find_dirs("/kaggle/input", {"defactify_subset"}, max_depth=12)
if not saved:
    saved = find_dirs("/kaggle/working", {"defactify_subset"}, max_depth=12)
assert saved, f"defactify_subset not found. Inputs: {os.listdir('/kaggle/input') if os.path.exists('/kaggle/input') else []}"

SAVED_ROOT = os.path.dirname(saved[0])
SPLITS_DIR = f"{SAVED_ROOT}/splits"
defactify = load_from_disk(f"{SAVED_ROOT}/defactify_subset")

# BigGAN (check saved subset first, fallback to attached input)
biggan_sub = f"{SAVED_ROOT}/biggan_subset"
if os.path.exists(biggan_sub):
    BIGGAN_DIR = biggan_sub
else:
    found = find_dirs("/kaggle/input", {"ai", "nature"})
    assert found, "Attach genimage-biggan"
    BIGGAN_DIR = os.path.commonpath(found)
    while os.path.basename(BIGGAN_DIR).lower() in {"ai", "nature", "train", "val"}:
        BIGGAN_DIR = os.path.dirname(BIGGAN_DIR)

# outputs
OUT = "/kaggle/working/AI_Image_Generalization"
CKPT_DIR, RESULTS_DIR = f"{OUT}/checkpoints", f"{OUT}/results"
os.makedirs(CKPT_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

print("Experiments:", sorted(os.listdir(SPLITS_DIR)))
print("BIGGAN_DIR:", BIGGAN_DIR)


In [ ]:
PIPELINE = "A"    # "A" = native 128 crop | "B" = resize short side to 128, then crop (ablation)
CROP, BATCH = 128, 128

class UnifiedImageDataset(Dataset):
    def __init__(self, csv_path, hf_data, local_root, transform=None, label_col="binary_label"):
        self.rows = pd.read_csv(csv_path).to_dict("records")
        self.hf, self.root, self.tf, self.label_col = hf_data, local_root, transform, label_col
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        if r["storage"] == "hf":
            img = self.hf[r["hf_split"]][int(r["hf_index"])][IMG_COL]
        else:
            img = Image.open(os.path.join(self.root, r["image_path"]))
        img = img.convert("RGB")
        if self.tf: img = self.tf(img)
        return img, int(r[self.label_col])

class JPEGRecompress:
    def __init__(self, q_min=65, q_max=95): self.q_min, self.q_max = q_min, q_max
    def __call__(self, img):
        buf = io.BytesIO()
        img.save(buf, "JPEG", quality=random.randint(self.q_min, self.q_max))
        buf.seek(0)
        return Image.open(buf).convert("RGB")

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
pre = [transforms.Resize(CROP)] if PIPELINE == "B" else []

train_tf = transforms.Compose(pre + [
    transforms.RandomCrop(CROP, pad_if_needed=True),
    transforms.RandomHorizontalFlip(),
    JPEGRecompress(65, 95),
    transforms.ToTensor(), norm])
eval_tf = transforms.Compose(pre + [
    transforms.CenterCrop(CROP),
    JPEGRecompress(90, 90),
    transforms.ToTensor(), norm])

def make_loader(csv, tf, shuffle):
    ds = UnifiedImageDataset(csv, defactify, BIGGAN_DIR, transform=tf)
    num_workers = 4 if os.name != 'nt' and not IS_TPU else 0
    return DataLoader(ds, batch_size=BATCH, shuffle=shuffle, num_workers=num_workers,
                      pin_memory=True, drop_last=shuffle)


In [ ]:
def build_resnet50(num_classes=2, freeze=True):
    m = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
    if freeze:
        for p in m.parameters(): p.requires_grad = False
    m.fc = nn.Linear(m.fc.in_features, num_classes)   # new head, always trainable
    return m


In [ ]:
@torch.no_grad()
def predict(model, loader):
    model.eval(); out = []
    
    if IS_TPU:
        import torch_xla.distributed.parallel_loader as pl
        para_loader = pl.ParallelLoader(loader, [device]).per_device_loader(device)
        for x, _ in para_loader:
            lo = model(x)
            out.append(torch.softmax(lo.float(), 1)[:, 1].cpu())
    else:
        use_amp = (device.type == "cuda")
        for x, _ in loader:
            if use_amp:
                with torch.autocast("cuda", dtype=torch.float16):
                    lo = model(x.to(device, non_blocking=True))
            else:
                lo = model(x.to(device))
            out.append(torch.softmax(lo.float(), 1)[:, 1].cpu())
            
    return torch.cat(out).numpy()

def test_metrics(df, p):
    y = df.binary_label.values
    res = {"auc": float(roc_auc_score(y, p)),
           "bal_acc": float(balanced_accuracy_score(y, p > 0.5)),
           "acc": float(((p > 0.5) == y).mean()),
           "n": int(len(df)), "by_group": {}, "auc_vs_matched_real": {}}
    for (src, gen), g in df.groupby(["source_dataset", "generator"]):
        i = g.index.values
        res["by_group"][f"{gen}|{src}"] = {"n": int(len(i)), "acc": float(((p[i] > 0.5) == y[i]).mean())}
    for gen in df[df.binary_label == 1].generator.unique():
        gi = df.index[df.generator == gen].values
        rsrc = "ImageNet_real" if gen == "BigGAN" else "Defactify_real"
        ri = df.index[df.source_dataset == rsrc].values
        if len(ri):
            yy = np.r_[np.ones(len(gi)), np.zeros(len(ri))]
            res["auc_vs_matched_real"][gen] = float(roc_auc_score(yy, np.r_[p[gi], p[ri]]))
    return res


In [ ]:
def train_run(exp, mode, epochs=5, lr=None, patience=2):
    tag = f"{exp}_{mode}_{PIPELINE}"
    res_path = f"{RESULTS_DIR}/{tag}.json"
    if os.path.exists(res_path):
        print("skip (done):", tag); return json.load(open(res_path))

    d = f"{SPLITS_DIR}/{exp}"
    tr_df, va_df, te_df = (pd.read_csv(f"{d}/{s}.csv") for s in ["train", "val", "test"])
    tr_loader = make_loader(f"{d}/train.csv", train_tf, True)
    va_loader = make_loader(f"{d}/val.csv", eval_tf, False)
    te_loader = make_loader(f"{d}/test.csv", eval_tf, False)

    # Class weights for dataset imbalance
    c = tr_df.binary_label.value_counts()
    w = torch.tensor([len(tr_df) / (2 * c[0]), len(tr_df) / (2 * c[1])], dtype=torch.float32, device=device)
    crit = nn.CrossEntropyLoss(weight=w)

    model = build_resnet50(2, freeze=(mode == "frozen")).to(device)
    lr = lr or (1e-3 if mode == "frozen" else 1e-4)
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    
    use_amp = (device.type == "cuda")
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    last, best_p = f"{CKPT_DIR}/{tag}_last.pt", f"{CKPT_DIR}/{tag}_best.pt"
    start, best, bad, hist = 0, -1.0, 0, []
    if os.path.exists(last):
        ck = torch.load(last, map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sched.load_state_dict(ck["sched"]); scaler.load_state_dict(ck["scaler"])
        start, best, bad, hist = ck["epoch"] + 1, ck["best"], ck["bad"], ck["hist"]
        print(f"resumed {tag} at epoch {start}")

    for ep in range(start, epochs):
        model.train()
        if mode == "frozen": model.eval()          # keep BN stats fixed
        t0, tl, n = time.time(), 0.0, 0

        if IS_TPU:
            import torch_xla.core.xla_model as xm
            import torch_xla.distributed.parallel_loader as pl
            para_tr_loader = pl.ParallelLoader(tr_loader, [device]).per_device_loader(device)
            for x, y in para_tr_loader:
                opt.zero_grad(set_to_none=True)
                loss = crit(model(x), y)
                loss.backward()
                xm.optimizer_step(opt)   # TPU barrier step
                tl += loss.item() * len(y); n += len(y)
        else:
            for x, y in tr_loader:
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                opt.zero_grad(set_to_none=True)
                if use_amp:
                    with torch.autocast("cuda", dtype=torch.float16):
                        loss = crit(model(x), y)
                    scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
                else:
                    loss = crit(model(x), y)
                    loss.backward(); opt.step()
                tl += loss.item() * len(y); n += len(y)
                
        sched.step()

        auc = roc_auc_score(va_df.binary_label, predict(model, va_loader))
        hist.append({"epoch": ep, "loss": tl / n, "val_auc": float(auc)})
        print(f"[{tag}] ep{ep}  loss={tl/n:.4f}  val_auc={auc:.4f}  ({time.time()-t0:.0f}s)")

        if auc > best:
            best, bad = auc, 0
            if IS_TPU:
                xm.save(model.state_dict(), best_p)
            else:
                torch.save(model.state_dict(), best_p)
        else:
            bad += 1
            
        save_obj = {"model": model.state_dict(), "opt": opt.state_dict(), "sched": sched.state_dict(),
                    "scaler": scaler.state_dict(), "epoch": ep, "best": best, "bad": bad, "hist": hist}
        if IS_TPU:
            xm.save(save_obj, last)
        else:
            torch.save(save_obj, last)
            
        if bad >= patience:
            print("early stop"); break

    model.load_state_dict(torch.load(best_p, map_location=device))
    res = test_metrics(te_df, predict(model, te_loader))
    res.update({"exp": exp, "mode": mode, "pipeline": PIPELINE, "best_val_auc": float(best), "hist": hist})
    json.dump(res, open(res_path, "w"), indent=2)
    print(f"TEST {tag}: AUC={res['auc']:.4f}  bal_acc={res['bal_acc']:.4f}")
    return res


In [ ]:
r = train_run("seen_all", "frozen", epochs=3)
print(json.dumps({k: r[k] for k in ["auc", "bal_acc", "auc_vs_matched_real", "by_group"]}, indent=2))


In [ ]:
EXPS = ["seen_all", "gan_to_diffusion", "diffusion_to_gan"] + \
       [f"unseen_{g}" for g in ["BigGAN", "DALL_E_3", "Midjourney_v6",
                                "Stable_Diffusion_2_1", "Stable_Diffusion_3", "Stable_Diffusion_XL"]]
for mode in ["frozen", "finetune"]:
    for exp in EXPS:
        train_run(exp, mode, epochs=5)


In [ ]:
rows = []
for f in Path(RESULTS_DIR).glob("*.json"):
    r = json.load(open(f))
    rows.append({"exp": r["exp"], "mode": r["mode"], "pipe": r["pipeline"],
                 "val_auc": r["best_val_auc"], "test_auc": r["auc"], "bal_acc": r["bal_acc"]})
tbl = pd.DataFrame(rows).sort_values(["pipe", "exp", "mode"])
display(tbl.round(4))
tbl.to_csv(f"{OUT}/summary.csv", index=False)
